# 03 EDA – dane po czyszczeniu

## 1. Konfiguracja

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from src.config import APA_STYLE, CYCLES, EDA_SUBREDDIT_STYLES, MAX_PER_WEEK, PHASE_ORDER, POLITICAL_SUBREDDITS, PROCESSED_DIR, STUDY_FILES

CLEAN_DIR = PROCESSED_DIR / "clean"
EDA_DIR = PROCESSED_DIR / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)
LOW_WEEK = 50  # bardzo słaby tydzień – próg tylko do opisu, nie kryterium wykluczenia
plt.rcParams.update(APA_STYLE)
plt.rcParams.update({"figure.dpi": 110})
file_labels = [f"{subreddit} {cycle}" for subreddit, cycle in STUDY_FILES]
political_labels = [f"{subreddit} {cycle}" for subreddit in POLITICAL_SUBREDDITS for cycle in CYCLES]


def draw_heatmap(ax, table, fmt, cmap, vmax=None):
    """Mapa cieplna z liczbami w komórkach (wiersze = indeks tabeli, kolumny = kolumny tabeli)."""
    values = table.to_numpy(dtype=float)
    vmax = vmax or np.nanmax(values)
    image = ax.imshow(values, cmap=cmap, vmin=0, vmax=vmax, aspect="auto")
    ax.set_xticks(range(table.shape[1]), [str(column) for column in table.columns], rotation=30, ha="right")
    ax.set_yticks(range(table.shape[0]), [str(index) for index in table.index])
    for row in range(values.shape[0]):
        for column in range(values.shape[1]):
            if not np.isnan(values[row, column]):
                ax.text(column, row, fmt.format(values[row, column]), ha="center", va="center", fontsize=7,
                        color="white" if values[row, column] > 0.6 * vmax else "black")
    return image

## 2. Wczytanie tabel

In [ ]:
weekly = pd.concat([pd.read_csv(PROCESSED_DIR / "weekly_counts" / f"r_{subreddit}_comments_{cycle}.csv", parse_dates=["week_start"])
                    .assign(subreddit=subreddit) for subreddit, cycle in STUDY_FILES], ignore_index=True)
funnel = pd.concat([pd.read_csv(PROCESSED_DIR / "funnel" / f"r_{subreddit}_comments_{cycle}.csv")
                    .assign(subreddit=subreddit, cycle=cycle) for subreddit, cycle in STUDY_FILES], ignore_index=True)
political = weekly[weekly["subreddit"].isin(POLITICAL_SUBREDDITS)]
print(weekly.shape, funnel.shape)

## 3. Tabela wykonalności (po czyszczeniu, ≥ 5 słów)

In [3]:
thresholds = [50, 100, 150, 200, 300]
grouped = weekly.groupby(["subreddit", "cycle", "phase"])["n_valid_5w"]
feasibility = grouped.agg(n_weeks="size", median="median", minimum="min")
for threshold in thresholds:
    feasibility[f"weeks_below_{threshold}"] = grouped.apply(lambda values: (values < threshold).sum())
feasibility.to_csv(EDA_DIR / "feasibility.csv")
feasibility.loc[POLITICAL_SUBREDDITS]

n_weeks   median  minimum  weeks_below_50  \
subreddit    cycle phase                                                      
democrats    2012  bazowa              12     31.0     14.0              11   
                   powyborcza           8    201.5    113.0               0   
                   przedwyborcza       64     83.5     24.0               7   
                   przejsciowa         11    152.0    102.0               0   
                   szczytowa            9    197.0    142.0               0   
...                                   ...      ...      ...             ...   
Conservative 2024  bazowa              22  22034.5  12059.0               0   
                   powyborcza           8  26420.0  12184.0               0   
                   przedwyborcza       52  16054.0   5776.0               0   
                   przejsciowa         11  11108.0   6846.0               0   
                   szczytowa           11  10781.0   6313.0               0   

                                  weeks_below_100  weeks_below_150  \
subreddit    cycle phase                                             
democrats    2012  bazowa                      12               12   
                   powyborcza                   0                1   
                   przedwyborcza               46               59   
                   przejsciowa                  0                5   
                   szczytowa                    0                1   
...                                           ...              ...   
Conservative 2024  bazowa                       0                0   
                   powyborcza                   0                0   
                   przedwyborcza                0                0   
                   przejsciowa                  0                0   
                   szczytowa                    0                0   

                                  weeks_below_200  weeks_below_300  
subreddit    cycle phase                                            
democrats    2012  bazowa                      12               12  
                   powyborcza                   4                6  
                   przedwyborcza               62               64  
                   przejsciowa                  8               10  
                   szczytowa                    6                9  
...                                           ...              ...  
Conservative 2024  bazowa                       0                0  
                   powyborcza                   0                0  
                   przedwyborcza                0                0  
                   przejsciowa                  0                0  
                   szczytowa                    0                0  

[80 rows x 8 columns]

### 3.1 Udział słabych tygodni

In [ ]:
low = (political.assign(low=political["n_valid_5w"] < LOW_WEEK).groupby(["subreddit", "cycle", "phase"])["low"].mean()
       .unstack("phase")[PHASE_ORDER])
low.index = [f"{subreddit} {cycle}" for subreddit, cycle in low.index]
low = low.reindex(political_labels)
between = (political.assign(between=(political["n_valid_5w"] >= LOW_WEEK) & (political["n_valid_5w"] < MAX_PER_WEEK))
           .groupby(["subreddit", "cycle", "phase"])["between"].mean().unstack("phase")[PHASE_ORDER])
between.index = [f"{subreddit} {cycle}" for subreddit, cycle in between.index]
between = between.reindex(political_labels)

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5), sharey=True)
image = draw_heatmap(axes[0], low, "{:.2f}", "Reds", vmax=1)
axes[0].set_title(f"Mniej niż {LOW_WEEK} komentarzy", loc="left")
fig.colorbar(image, ax=axes[0], label="Udział tygodni fazy")
image = draw_heatmap(axes[1], between, "{:.2f}", "Blues", vmax=1)
axes[1].set_title(f"Od {LOW_WEEK} do {MAX_PER_WEEK - 1} komentarzy", loc="left")
fig.colorbar(image, ax=axes[1], label="Udział tygodni fazy")
fig.suptitle("Udział słabych tygodni po czyszczeniu (komentarze ≥ 5 słów, subreddit i cykl × faza)", x=0.01, ha="left", fontsize=12)
plt.tight_layout()
plt.show()
print(f"tygodnie polityczne < {LOW_WEEK}:", int((political["n_valid_5w"] < LOW_WEEK).sum()),
      f"| < {MAX_PER_WEEK}:", int((political["n_valid_5w"] < MAX_PER_WEEK).sum()), "| wszystkie:", len(political))

**Jak czytać**
- Wiersz to subreddit w jednym cyklu, kolumna to faza. Liczba w komórce to udział tygodni fazy, w których po czyszczeniu zostało mniej niż 50 komentarzy od 5 słów (lewy panel) albo od 50 do 299 (prawy panel). Ciemniejszy kolor to więcej słabych tygodni.

**Co widać**
- Poniżej 300 komentarzy ≥ 5 słów jest 309 z 1672 tygodni subredditów politycznych (przed czyszczeniem 281).
  Przybyło głównie w 2016: r/democrats 16 → 23, r/Liberal 11 → 16, r/Republican 7 → 10. W 2012 przybyło po 1–2 tygodnie na plik.
- Poniżej 50 jest 42 tygodnie: r/Liberal 2012 (19), r/democrats 2012 (18), r/Republican 2012 (4) i jeden tydzień r/Republican 2020
  (pusty tydzień 17.08.2020).
- Od 2020 poniżej 300 schodzą tylko r/Liberal (22 tygodnie w 2020, 6 w 2024) i r/Republican 2020 (3).
- Czyszczenie nie tworzy nowych problemów z liczebnością. Słabe komórki to nadal cykl 2012 w r/democrats, r/Liberal i r/Republican, zwłaszcza faza bazowa i przedwyborcza.

## 4. Lejek

In [8]:
funnel_total = funnel.pivot_table(index=["subreddit", "cycle"], columns="step", values="dropped", sort=False)
funnel_total.to_csv(EDA_DIR / "funnel_total.csv")
funnel_total

step                    uszkodzone_linie  poza_oknem  f_deleted  f_moderator  \
subreddit        cycle                                                         
democrats        2012                0.0         0.0     1009.0         20.0   
                 2016                0.0         0.0    11452.0       2301.0   
                 2020                0.0         0.0    73617.0      13563.0   
                 2024                0.0         0.0   291992.0      23806.0   
Liberal          2012                0.0         0.0     1829.0         50.0   
                 2016                0.0         0.0     4995.0        415.0   
                 2020                0.0         0.0    17791.0        131.0   
                 2024                0.0         0.0    37071.0        424.0   
Republican       2012                0.0         0.0     8520.0         60.0   
                 2016                0.0         0.0    12629.0       1460.0   
                 2020                0.0         0.0    57815.0      25907.0   
                 2024                0.0         0.0    60620.0      23098.0   
Conservative     2012                0.0         0.0    29088.0        846.0   
                 2016                2.0         0.0   103192.0       6369.0   
                 2020                0.0         0.0  1597621.0      22912.0   
                 2024                0.0         0.0  2232331.0       5017.0   
running          2012                0.0         0.0    19129.0         64.0   
                 2016                0.0         0.0    20764.0       4969.0   
                 2020                0.0         0.0    46140.0      10901.0   
                 2024                0.0         0.0     5777.0      62709.0   
                 2014                0.0         0.0    13141.0        213.0   
                 2018                0.0         0.0     9820.0       3463.0   
                 2022                0.0         0.0    19679.0      11543.0   
languagelearning 2012                0.0         0.0     2944.0          8.0   
                 2016                0.0         0.0     7122.0       1914.0   
                 2020                0.0        22.0    29323.0      11295.0   
                 2024                0.0        44.0    10019.0      31663.0   
                 2014                0.0         0.0     4230.0         73.0   
                 2018                0.0         0.0     6705.0       2080.0   
                 2022                0.0         0.0    19022.0       9926.0   

step                     f_bot  f_high_volume  f_no_text   f_media  \
subreddit        cycle                                               
democrats        2012      9.0            0.0       76.0     957.0   
                 2016    685.0            0.0      928.0    6007.0   
                 2020   1312.0            0.0     1092.0    7263.0   
                 2024     19.0            0.0     4545.0   24745.0   
Liberal          2012     52.0            0.0      118.0    1656.0   
                 2016     17.0            0.0      358.0    3063.0   
                 2020     19.0            0.0      262.0    2409.0   
                 2024     12.0            0.0      323.0    3038.0   
Republican       2012      5.0            0.0      445.0    6156.0   
                 2016    152.0            0.0     1000.0    7529.0   
                 2020    296.0            0.0     1323.0    8613.0   
                 2024      9.0            0.0     1079.0    5396.0   
Conservative     2012     14.0            0.0     1950.0   20637.0   
                 2016     68.0            0.0     6056.0   38354.0   
                 2020    965.0            0.0    22927.0  148607.0   
                 2024     10.0            0.0     7760.0   47622.0   
running          2012      1.0            0.0     1011.0   22919.0   
                 2016    255.0            0.0     1775.0   44355.0   
                 2020    639.0         

In [ ]:
steps = [step for step in funnel["step"].unique() if step not in ("uszkodzone_linie", "poza_oknem")]
inside = funnel[funnel["step"] == "poza_oknem"].set_index(["subreddit", "cycle"])["remaining"]
share_dropped = funnel[funnel["step"].isin(steps)].pivot_table(index=["subreddit", "cycle"], columns="step", values="dropped", sort=False)
share_dropped = share_dropped[steps].div(inside, axis=0)
share_dropped.index = [f"{subreddit} {cycle}" for subreddit, cycle in share_dropped.index]
share_dropped = share_dropped.reindex(file_labels)
share_dropped.to_csv(EDA_DIR / "funnel_share_dropped.csv")
fig, ax = plt.subplots(figsize=(10, 9))
left = np.zeros(len(share_dropped))
for position, step in enumerate(steps):
    ax.barh(share_dropped.index, share_dropped[step], left=left, label=step, color=f"C{position}")
    left += share_dropped[step].to_numpy()
ax.invert_yaxis()
ax.set_xlabel("Udział rekordów okna odrzuconych na danym kroku")
ax.set_title("Lejek czyszczenia: jaka część rekordów każdego pliku odpada na kolejnych krokach", loc="left")
ax.legend(ncol=4, loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

**Jak czytać**
- Jeden słupek to plik. Kolorowe odcinki to udział rekordów okna odrzuconych na kolejnym kroku czyszczenia (kolor oznacza krok, kroki idą w kolejności lejka). Długość całego słupka to łączny udział odrzuconych rekordów.

**Co widać**
- Odrzucono 14–16% rekordów w subredditach politycznych w 2012, 15–20% w 2016, 27–28% w 2020 i 26–30% w 2024. Wyjątkiem jest r/Conservative 2024 z 52%.
- We wszystkich 16 plikach politycznych największym krokiem jest `f_deleted`. W subredditach odniesienia do 2018 roku dominuje `f_media`,
  w 2020 i 2022 `f_deleted`, a w 2024 `f_moderator`.
- `f_high_volume` nie odrzuca niczego, bo próg `HIGH_VOLUME_MIN_COMMENTS` nie jest jeszcze ustawiony. `f_not_english` ma znaczenie tylko w r/languagelearning.
- Wzrost odrzuceń z cyklu na cykl w polityce wynika prawie wyłącznie z usuniętych komentarzy. Pozostałe kroki mają podobny rozmiar we wszystkich cyklach.

## 5. Długość tekstów po czyszczeniu

In [ ]:
length_rows, histograms = [], {}
for subreddit, cycle in tqdm(STUDY_FILES):
    clean = pd.read_parquet(CLEAN_DIR / f"r_{subreddit}_comments_{cycle}.parquet", columns=["n_words", "keep"])
    kept_words = clean.loc[clean["keep"], "n_words"].to_numpy()
    length_rows.append({"subreddit": subreddit, "cycle": cycle, "n_kept": len(kept_words), "mean": kept_words.mean(),
                        "p25": np.percentile(kept_words, 25), "median": np.median(kept_words), "p75": np.percentile(kept_words, 75),
                        "p90": np.percentile(kept_words, 90), "share_below_5w": (kept_words < 5).mean(),
                        "share_below_10w": (kept_words < 10).mean()})
    histograms[(subreddit, cycle)] = np.bincount(np.clip(kept_words, 0, 100), minlength=101)
length = pd.DataFrame(length_rows)
length.to_csv(EDA_DIR / "length.csv", index=False)
length.round(3)

In [ ]:
fig, axes = plt.subplots(1, len(CYCLES), figsize=(14, 4), sharey=True)
for ax, cycle in zip(axes, CYCLES):
    for subreddit in POLITICAL_SUBREDDITS:
        counts = histograms[(subreddit, cycle)]
        color, linestyle = EDA_SUBREDDIT_STYLES[subreddit]
        ax.plot(np.arange(1, 100), (np.cumsum(counts) / counts.sum())[1:100], color=color, linestyle=linestyle, label=f"r/{subreddit}")
    ax.axvline(5, color="0.3", linestyle=":", linewidth=0.8)
    ax.axvline(10, color="0.3", linestyle="--", linewidth=0.8)
    ax.set_xscale("log")
    ax.set_title(f"Cykl {cycle}", loc="left")
    ax.set_xlabel("Liczba słów")
axes[0].set_ylabel("Udział komentarzy o tej lub mniejszej długości")
axes[0].legend(fontsize=8)
fig.suptitle("Długość komentarzy po czyszczeniu: skumulowany udział według liczby słów (kropkowana = próg 5 słów, przerywana = próg 10 słów)",
             x=0.01, ha="left", fontsize=12)
plt.tight_layout()
plt.show()

**Jak czytać**
- Jeden panel na cykl, linia na subreddit. Oś X (skala logarytmiczna): liczba słów w komentarzu. Oś Y: udział komentarzy o tej lub mniejszej długości. Im szybciej krzywa rośnie, tym więcej krótkich komentarzy.
- Kropkowana linia pionowa to próg 5 słów, przerywana to próg 10 słów.

**Co widać**
- Mediana długości w subredditach politycznych spada z 24–28 słów w 2012 do 16–20 w 2020 i 2024. Udział komentarzy < 10 słów rośnie z 18–21% do 25–30%.
- W r/running mediana rośnie (27 → 32 słowa), a w r/languagelearning jest stała (26–29).
- Dystrybuanty czterech subredditów politycznych w jednym cyklu prawie się pokrywają. Różnice między cyklami są większe niż między obozami;
  wyjątkiem są nieco dłuższe komentarze w r/Liberal 2024.
- Skracanie komentarzy dotyczy polityki, nie całego Reddita. Próg 5 słów (linia kropkowana) obcina 7–13% komentarzy, próg 10 słów (przerywana) 18–30%.

## 6. Podsumowanie

In [ ]:
summary = pd.DataFrame({
    "pliki": [len(STUDY_FILES)],
    "rekordy_w_oknie": [int(inside.sum())],
    "rekordy_po_czyszczeniu": [int(weekly["n_keep"].sum())],
    "po_czyszczeniu_>=5_slow": [int(weekly["n_valid_5w"].sum())],
    f"tygodnie_polityczne_<{LOW_WEEK}": [int((political["n_valid_5w"] < LOW_WEEK).sum())],
    f"tygodnie_polityczne_<{MAX_PER_WEEK}": [int((political["n_valid_5w"] < MAX_PER_WEEK).sum())],
    "tygodnie_polityczne": [len(political)],
})
summary["udzial_zachowanych"] = summary["rekordy_po_czyszczeniu"] / summary["rekordy_w_oknie"]
summary.to_csv(EDA_DIR / "clean_summary.csv", index=False)
summary

,pliki,rekordy_w_oknie,rekordy_po_czyszczeniu,po_czyszczeniu_>=5_slow,tygodnie_polityczne_<50,tygodnie_polityczne_<300,tygodnie_polityczne,udzial_zachowanych
0,30,20577989,14839459,13465220,42,309,1672,0.721133


: 